# SD-3.5 Diffusion-DPO pipeline (rl_aes / rl_tech)

Runs the `sd35-dpo` repo end to end on Colab A100: preference pairs from metric rewards → Diffusion-DPO LoRA on top of the plain personalisation LoRA → survey images (`generated_rl_aes/`, `generated_rl_tech/`) in the bucket layout.

Order: **0 setup → 1 smoke test (CPU) → 2 prompts → 3 tiny GPU run → 4 full runs → 5 triplets → 6 upload**. Save to Drive after every stage.

## 0. Setup

In [ ]:
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
import os
REPO = '/content/sd35-dpo'
DRIVE = '/content/drive/MyDrive/Doktorat/content'
# unzip the repo from Drive (or git clone)
!rm -rf {REPO} && unzip -q {DRIVE}/sd35-dpo.zip -d /content
%cd {REPO}
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao 2>/dev/null
from huggingface_hub import login
from google.colab import userdata
login(userdata.get('HF_TOKEN'))

## 1. Smoke test on CPU (tiny random SD3 transformer, no downloads)

In [ ]:
!python tests/smoke_tiny.py

## 2. Prompts = the 178 catalogue prompts (same distribution as the survey)

In [ ]:
CATALOG = f'{DRIVE}/mos-eval/tasks_catalog.json'   # copy of mos-eval/data/tasks_catalog.json
!python scripts/prompts_from_catalog.py {CATALOG} prompts.txt
!head -3 prompts.txt > p3.txt

## 3. Tiny GPU run (3 prompts, 5 DPO steps) — verifies SD-3.5 loading, fused plain LoRA, reward, training, saving

In [ ]:
!sed -i 's#^plain_lora:.*#plain_lora: /content/drive/MyDrive/Doktorat/content/results/merged_finetune/plain#' configs/*.yaml
!python scripts/run_pipeline.py --config configs/aesthetic.yaml --prompts p3.txt --max-steps 5

## 4. Full runs (resumable: existing images in `pairs/images` are re-used)

In [ ]:
!python scripts/run_pipeline.py --config configs/aesthetic.yaml --prompts prompts.txt

In [ ]:
!python scripts/run_pipeline.py --config configs/technical.yaml --prompts prompts.txt

### Training curve

In [ ]:
import json, matplotlib.pyplot as plt
for ax in ['rl_aes','rl_tech']:
    p=f'{DRIVE}/results/{ax}/lora/train_log.json'
    if not os.path.exists(p): continue
    h=json.load(open(p))['history']
    plt.plot([x['step'] for x in h],[x['implicit_acc'] for x in h],label=f'{ax} implicit_acc')
plt.axhline(0.5,ls='--',c='gray'); plt.legend(); plt.xlabel('step'); plt.show()

## 5. Survey images — same prompt + seed per method

Match `--steps/--guidance` to the settings used for the existing `plain` images.

In [ ]:
OUT = '/content/drive/MyDrive/rufgen'
PLAIN = f'{DRIVE}/results/merged_finetune/plain'
for m in ['rl_aes','rl_tech']:
    !python eval/generate_triplets.py --method {m} --catalog {CATALOG} --plain-lora {PLAIN} \
        --dpo-lora {DRIVE}/results/{m}/lora --out {OUT} --seeds 0-6 --steps 28 --guidance 7.0

### Quick look: plain vs rl_aes vs rl_tech

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
tasks=json.load(open(CATALOG))['tasks'][:4]
fig,axs=plt.subplots(len(tasks),3,figsize=(12,4*len(tasks)))
for i,t in enumerate(tasks):
    m=t['meta']
    for j,(meth,pre) in enumerate([('plain','generated'),('rl_aes','generated_rl_aes'),('rl_tech','generated_rl_tech')]):
        p=f"{OUT}/{pre}/{m['model']}/{m['color']}/{m['top']}/seed_{m['seed']:04d}.png"
        if os.path.exists(p): axs[i,j].imshow(Image.open(p))
        axs[i,j].set_title(f"{t['id']} {meth}",fontsize=8); axs[i,j].axis('off')
plt.tight_layout(); plt.show()

## 6. Upload to the bucket (URLs in `tasks_catalog.json` already point here)

In [ ]:
from google.colab import auth; auth.authenticate_user()
!gsutil -m rsync -r {OUT}/generated_rl_aes  gs://ruf-ai/rufgen/generated_rl_aes
!gsutil -m rsync -r {OUT}/generated_rl_tech gs://ruf-ai/rufgen/generated_rl_tech